# Experiment 9: Regression Model to Predict House Prices

**Objective:** Predict California housing prices using multiple regression models (Linear Regression, Ridge Regression, and Random Forest Regressor), optimize hyperparameters with GridSearchCV, and evaluate with 5-fold cross-validation.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1) Load dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')  # target units ~ 100k USD

print("Dataset shape:", X.shape)
print("Top correlations with target:
", X.join(y).corr()['MedHouseVal'].sort_values(ascending=False))


In [ ]:
# 2) Train/Test split & Scale
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1) if (n - p - 1) > 0 else np.nan

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    y_pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, y_pred)
    adj = adjusted_r2(r2, y_te.shape[0], X_tr.shape[1])
    cv_scores = cross_val_score(model, X_tr, y_tr, scoring='neg_mean_squared_error', cv=5, n_jobs=-1)
    cv_rmse = np.sqrt(-cv_scores).mean()
    
    print(f"---- {name} ----")
    print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f} | AdjR2: {adj:.4f} | CV RMSE: {cv_rmse:.4f}
")
    return {'name': name, 'mae': mae, 'mse': mse, 'rmse': rmse, 'r2': r2, 'adj_r2': adj, 'y_pred': y_pred}


In [ ]:
# 3) Baseline: Linear Regression
lin = LinearRegression().fit(X_train_scaled, y_train)
res_lin = evaluate_model("LinearRegression", lin, X_train_scaled, X_test_scaled, y_train, y_test)


In [ ]:
# 4) Ridge Regression (GridSearchCV)
gs_ridge = GridSearchCV(Ridge(random_state=RANDOM_STATE), {'alpha': [0.1, 1.0, 10.0, 50.0]},
                        scoring='neg_mean_squared_error', cv=5, n_jobs=-1)
gs_ridge.fit(X_train_scaled, y_train)
best_ridge = gs_ridge.best_estimator_
res_ridge = evaluate_model("Ridge (best)", best_ridge, X_train_scaled, X_test_scaled, y_train, y_test)


In [ ]:
# 5) Random Forest Regressor (GridSearchCV)
gs_rf = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE),
                     {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
                     scoring='neg_mean_squared_error', cv=3, n_jobs=-1)
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_
res_rf = evaluate_model("RandomForest (best)", best_rf, X_train.values, X_test.values, y_train, y_test)


In [ ]:
# 6) Compare & Choose Best Model
results = pd.DataFrame([
    {'model': res_lin['name'], 'rmse': res_lin['rmse'], 'r2': res_lin['r2']},
    {'model': res_ridge['name'], 'rmse': res_ridge['rmse'], 'r2': res_ridge['r2']},
    {'model': res_rf['name'], 'rmse': res_rf['rmse'], 'r2': res_rf['r2']}
])
print("Summary comparison:
", results.sort_values('rmse'))

best_model_name = results.loc[results['rmse'].idxmin(), 'model']
print("
Best model on test RMSE:", best_model_name)

if best_model_name.startswith("RandomForest"):
    best_model, y_pred = best_rf, res_rf['y_pred']
    joblib.dump({'model': best_model, 'scaler': None, 'features': list(X.columns)}, "best_house_price_model.joblib")
elif best_model_name.startswith("Ridge"):
    best_model, y_pred = best_ridge, res_ridge['y_pred']
    joblib.dump({'model': best_model, 'scaler': scaler, 'features': list(X.columns)}, "best_house_price_model.joblib")
else:
    best_model, y_pred = lin, res_lin['y_pred']
    joblib.dump({'model': best_model, 'scaler': scaler, 'features': list(X.columns)}, "best_house_price_model.joblib")
print("Saved best model to 'best_house_price_model.joblib'")


In [ ]:
# 7) Diagnostic Plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5)
minv, maxv = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([minv, maxv], [minv, maxv], '--', color='red', label='Ideal')
plt.xlabel("Actual median value")
plt.ylabel("Predicted median value")
plt.title(f"Actual vs Predicted ({best_model_name})")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_test.values - y_pred
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, linestyle='--', color='black')
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title(f"Residuals vs Predicted ({best_model_name})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
